# Test Heat Load Calculation

This notebook tests the heat load calculation function with sample JSON data.

## Steps:
1. Load required modules
2. Create sample JSON data
3. Test the calculation function
4. Display results

In [ ]:
%pip install pandas psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import calculation functions
from heat_load_wrapper import calculate_heat_load_from_json
from heat_load_utils import get_u_values, get_t_design, get_air_change_rate, get_n_walls_touching

# Get database URL
DATABASE_URL = os.getenv('DATABASE_URL')

print(f"Database URL: {'✓ Set' if DATABASE_URL else '✗ Not set'}")

Database URL: ✓ Set


## Sample JSON Data

Create sample JSON data for testing. This includes all required fields and some optional ones.

In [8]:
# Sample JSON data for a renovated building
sample_json = {
    # Required fields
    "area": 100.0,              # Heated floor area per floor (m²)
    "N_f": 1,                   # Number of floors
    "year": 1986,               # Construction year
    "postal_code": 60315,       # Postal code for design temperature lookup
    
    # Optional: Renovation information
    "renovated": True,
    "renovations": {
        "windows": True,
        "roof": False,
        "walls": False,
        "floor": True
    },
    "window_replacement_year": 2010,
    "roof_insulated": True,
    "walls_insulated": True,
    
    # Optional: Number of walls touching other buildings
    "n_walls_touching": 0,
    
    # Optional: Correction factors (defaults will be used if not specified)
    "h": 2.5,                   # Height of each floor (m)
    "f_floor": 0.5,
    "f_wall": 1.0,
    "f_roof": 1.0,
    "f_window": 1.0,
    "f_wall_touching": 0.5,
    "t_indoor": 21.0,           # Indoor design temperature (°C)
    "is_ground_floor": True,
    "is_top_floor": True,
    
    # Optional: Fallback U-values (used if database lookup fails)
    # "u_values": {
    #     "floor": 0.3,
    #     "wall": 0.4,
    #     "roof": 0.25,
    #     "window": 1.2
    # }
}

# Display the JSON
print("Sample JSON Data:")
print(json.dumps(sample_json, indent=2))

Sample JSON Data:
{
  "area": 100.0,
  "N_f": 1,
  "year": 1986,
  "postal_code": 60315,
  "renovated": true,
  "renovations": {
    "windows": true,
    "roof": false,
    "walls": false,
    "floor": true
  },
  "window_replacement_year": 2010,
  "roof_insulated": true,
  "walls_insulated": true,
  "n_walls_touching": 0,
  "h": 2.5,
  "f_floor": 0.5,
  "f_wall": 1.0,
  "f_roof": 1.0,
  "f_window": 1.0,
  "f_wall_touching": 0.5,
  "t_indoor": 21.0,
  "is_ground_floor": true,
  "is_top_floor": true
}


In [9]:
# Reload module to ensure latest code is used
import importlib
import heat_load_utils
importlib.reload(heat_load_utils)
print("✓ Module reloaded")

✓ Module reloaded


## Test Individual Utility Functions

Test the utility functions separately to verify they work correctly.

In [10]:
if DATABASE_URL:
    print("Testing utility functions:")
    print("=" * 60)
    
    # Test get_air_change_rate
    n = get_air_change_rate(sample_json)
    print(f"✓ Air change rate (n): {n} 1/h")
    
    # Test get_n_walls_touching
    n_walls = get_n_walls_touching(sample_json)
    print(f"✓ Walls touching: {n_walls}")
    
    # Test get_t_design
    try:
        t_design = get_t_design(sample_json, DATABASE_URL)
        print(f"✓ Design temperature (t_design): {t_design:.2f}°C")
    except Exception as e:
        print(f"✗ Error getting design temperature: {e}")
        t_design = None
    
    # Test get_u_values
    # Reload module to ensure latest code is used
    import importlib
    import heat_load_utils
    importlib.reload(heat_load_utils)
    from heat_load_utils import get_u_values
    
    try:
        u_values = get_u_values(sample_json, DATABASE_URL)
        print(f"\n✓ U-values from database:")
        print(f"  U_floor: {u_values['U_floor']} W/(m²·K)")
        print(f"  U_wall: {u_values['U_wall']} W/(m²·K)")
        print(f"  U_roof: {u_values['U_roof']} W/(m²·K)")
        print(f"  U_window: {u_values['U_window']} W/(m²·K)")
    except Exception as e:
        print(f"✗ Error getting U-values: {e}")
        u_values = None
else:
    print("⚠ DATABASE_URL not set. Skipping database-dependent tests.")
    print("Using fallback U-values from JSON...")
    u_values = sample_json.get('u_values', {})
    u_values = {
        'U_floor': u_values.get('floor', 0.3),
        'U_wall': u_values.get('wall', 0.4),
        'U_roof': u_values.get('roof', 0.25),
        'U_window': u_values.get('window', 1.2)
    }
    print(f"U-values: {u_values}")

Testing utility functions:
✓ Air change rate (n): 0.5 1/h
✓ Walls touching: 0
✓ Design temperature (t_design): -10.40°C

✓ U-values from database:
  U_floor: 0.6 W/(m²·K)
  U_wall: 0.35 W/(m²·K)
  U_roof: 0.35 W/(m²·K)
  U_window: 1.3 W/(m²·K)


## Save Sample JSON to File

Save the sample JSON to a file for testing.

In [ ]:
# Save sample JSON to file
sample_json_path = Path('sample_building.json')
with open(sample_json_input, 'w') as f:
    json.dump(sample_json, f, indent=2)

print(f"✓ Sample JSON saved to: {sample_json_input.absolute()}")

✓ Sample JSON saved to: /home/abhishek/Documents/heatpump_ai_2/calculation_functions/sample_building.json


## Calculate Heat Load

Run the complete heat load calculation using the wrapper function.

In [13]:
if DATABASE_URL:
    try:
        # Reload modules to ensure latest code is used
        import importlib
        import heat_load
        import heat_load_wrapper
        importlib.reload(heat_load)
        importlib.reload(heat_load_wrapper)
        from heat_load_wrapper import calculate_heat_load_from_json
        
        # Calculate heat load using the wrapper function
        result = calculate_heat_load_from_json(
            json_input=sample_json_path,
            database_url=DATABASE_URL
        )
        
        print("Heat Load Calculation Results:")
        print("=" * 60)
        print(f"Total Heat Load: {result['heat_load']:.2f} kW")
        print(f"\nTransmission Heat Transfer Coefficients (H_t):")
        print(f"  Total H_t: {result['H_t']:.2f} W/K")
        print(f"  H_t_floor: {result['H_t_floor']:.2f} W/K")
        print(f"  H_t_wall: {result['H_t_wall']:.2f} W/K")
        print(f"  H_t_roof: {result['H_t_roof']:.2f} W/K")
        print(f"  H_t_window: {result['H_t_window']:.2f} W/K")
        print(f"\nVentilation Heat Transfer Coefficient (H_v):")
        print(f"  H_v: {result['H_v']:.2f} W/K")
        print(f"  Volume per floor (V): {result['V']:.2f} m³")
        print(f"  Total volume (V_total): {result['V_total']:.2f} m³")
        print(f"  Air change rate (n): {result['n']:.2f} 1/h")
        print(f"\nTemperature Difference (Δt): {result['Dt']:.2f} K")
        
        print("\n" + "=" * 60)
        print("\nDetailed Breakdown:")
        # Calculate heat load in W first, then convert to kW
        heat_load_w = (result['H_t'] + result['H_v']) * result['Dt']
        print(f"  Heat Load (W) = (H_t + H_v) × Δt")
        print(f"  Heat Load (W) = ({result['H_t']:.2f} + {result['H_v']:.2f}) × {result['Dt']:.2f}")
        print(f"  Heat Load (W) = {heat_load_w:.2f} W")
        print(f"  Heat Load (kW) = {heat_load_w / 1000:.2f} kW")
        print(f"\n  H_t breakdown:")
        print(f"    H_t = H_t_floor + H_t_wall + H_t_roof + H_t_window")
        print(f"    H_t = {result['H_t_floor']:.2f} + {result['H_t_wall']:.2f} + {result['H_t_roof']:.2f} + {result['H_t_window']:.2f}")
        print(f"    H_t = {result['H_t']:.2f} W/K")
        print(f"\n  H_v breakdown:")
        print(f"    H_v = 0.34 × n × V × N_f")
        print(f"    H_v = 0.34 × {result['n']:.2f} × {result['V']:.2f} × {result['N_f']}")
        print(f"    H_v = 0.34 × {result['n']:.2f} × {result['V_total']:.2f}")
        print(f"    H_v = {result['H_v']:.2f} W/K")
        
    except Exception as e:
        print(f"✗ Error during calculation: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠ DATABASE_URL not set. Cannot run calculation.")
    print("Please set DATABASE_URL environment variable to test the calculation.")

Heat Load Calculation Results:
Total Heat Load: 6.92 kW

Transmission Heat Transfer Coefficients (H_t):
  Total H_t: 178.00 W/K
  H_t_floor: 30.00 W/K
  H_t_wall: 35.00 W/K
  H_t_roof: 35.00 W/K
  H_t_window: 78.00 W/K

Ventilation Heat Transfer Coefficient (H_v):
  H_v: 42.50 W/K
  Volume per floor (V): 250.00 m³
  Total volume (V_total): 250.00 m³
  Air change rate (n): 0.50 1/h

Temperature Difference (Δt): 31.40 K


Detailed Breakdown:
  Heat Load (W) = (H_t + H_v) × Δt
  Heat Load (W) = (178.00 + 42.50) × 31.40
  Heat Load (W) = 6923.90 W
  Heat Load (kW) = 6.92 kW

  H_t breakdown:
    H_t = H_t_floor + H_t_wall + H_t_roof + H_t_window
    H_t = 30.00 + 35.00 + 35.00 + 78.00
    H_t = 178.00 W/K

  H_v breakdown:
    H_v = 0.34 × n × V × N_f
    H_v = 0.34 × 0.50 × 250.00 × 1
    H_v = 0.34 × 0.50 × 250.00
    H_v = 42.50 W/K


## Test with Different Scenarios

Test the calculation with different building configurations.

In [ ]:
if DATABASE_URL:
    # Test scenario 1: Non-renovated building
    print("Scenario 1: Non-renovated building")
    print("-" * 60)
    scenario1 = sample_json.copy()
    scenario1['renovated'] = False
    scenario1['renovations'] = {}
    
    scenario1_path = Path('scenario1_non_renovated.json')
    with open(scenario1_path, 'w') as f:
        json.dump(scenario1, f, indent=2)
    
    try:
        result1 = calculate_heat_load_from_json(scenario1_path, DATABASE_URL)
        print(f"  Heat Load: {result1['heat_load']:.2f} kW")
        print(f"  H_t: {result1['H_t']:.2f} W/K")
        print(f"  H_v: {result1['H_v']:.2f} W/K")
    except Exception as e:
        print(f"  ✗ Error: {e}")
    
    print("\n" + "=" * 60)
    
    # Test scenario 2: Larger building
    print("Scenario 2: Larger building (200 m², 3 floors)")
    print("-" * 60)
    scenario2 = sample_json.copy()
    scenario2['area'] = 200.0
    scenario2['N_f'] = 3
    
    scenario2_path = Path('scenario2_larger.json')
    with open(scenario2_path, 'w') as f:
        json.dump(scenario2, f, indent=2)
    
    try:
        result2 = calculate_heat_load_from_json(scenario2_path, DATABASE_URL)
        print(f"  Heat Load: {result2['heat_load']:.2f} kW")
        print(f"  H_t: {result2['H_t']:.2f} W/K")
        print(f"  H_v: {result2['H_v']:.2f} W/K")
    except Exception as e:
        print(f"  ✗ Error: {e}")
else:
    print("⚠ DATABASE_URL not set. Skipping scenario tests.")

Scenario 1: Non-renovated building
------------------------------------------------------------


: 